In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', names=["label", "text"])

print(df)

     label                                               text
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham               Will ü b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [3]:
df["label"] = df["label"].map({"ham": 0, "spam": 1})

In [4]:
print(df)

      label                                               text
0         0  Go until jurong point, crazy.. Available only ...
1         0                      Ok lar... Joking wif u oni...
2         1  Free entry in 2 a wkly comp to win FA Cup fina...
3         0  U dun say so early hor... U c already then say...
4         0  Nah I don't think he goes to usf, he lives aro...
...     ...                                                ...
5567      1  This is the 2nd time we have tried 2 contact u...
5568      0               Will ü b going to esplanade fr home?
5569      0  Pity, * was in mood for that. So...any other s...
5570      0  The guy did some bitching but I acted like i'd...
5571      0                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df["text"] = df["text"].apply(clean_text)

In [6]:
print(df)

      label                                               text
0         0  go until jurong point  crazy   available only ...
1         0                      ok lar    joking wif u oni   
2         1  free entry in   a wkly comp to win fa cup fina...
3         0  u dun say so early hor    u c already then say   
4         0  nah i don t think he goes to usf  he lives aro...
...     ...                                                ...
5567      1  this is the  nd time we have tried   contact u...
5568      0               will   b going to esplanade fr home 
5569      0  pity    was in mood for that  so   any other s...
5570      0  the guy did some bitching but i acted like i d...
5571      0                         rofl  its true to its name

[5572 rows x 2 columns]


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["text"])
y = df["label"]

In [8]:
print(X.shape)
print(y.head())

(5572, 7759)
0    0
1    0
2    1
3    0
4    0
Name: label, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_model = LogisticRegression()
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))


Logistic Regression Accuracy: 0.9542600896860987


In [13]:
print("\nClassification of Log_Model's Report:\n")
print(classification_report(y_test, y_pred_log))


Classification of Log_Model's Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97       962
           1       0.98      0.68      0.80       153

    accuracy                           0.95      1115
   macro avg       0.97      0.84      0.89      1115
weighted avg       0.96      0.95      0.95      1115



In [14]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))


Naive Bayes Accuracy: 0.957847533632287


In [15]:
print("\nClassification of Naive_Model's Report:\n")
print(classification_report(y_test, y_pred_nb))


Classification of Naive_Model's Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.98       962
           1       1.00      0.69      0.82       153

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.95      1115



In [16]:
from sklearn.svm import SVC

model = SVC()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred))

SVM Accuracy: 0.9739910313901345


In [17]:
print("\nClassificationof SVM_Model's  Report:\n")
print(classification_report(y_test, y_pred))


Classificationof SVM_Model's  Report:

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       962
           1       0.98      0.82      0.90       153

    accuracy                           0.97      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [18]:
#LOG
new_text = ["Urgent! Your account has been suspended. Click the link to verify now"]

new_text_vec = vectorizer.transform(new_text)

prediction = log_model.predict(new_text_vec)

if prediction[0] == 1:
    print("Spam XX")
else:
    print("Ham !!")

Ham !!


In [19]:
#Naive
new_text = ["Congratulations you won a lottery click here"]

new_text_vec = vectorizer.transform(new_text)

prediction = nb_model.predict(new_text_vec)

if prediction[0] == 1:
    print("Spam XX")
else:
    print("Ham !!")

Ham !!


In [21]:
#SVM
new_text = ["Free entry in a £1000 prize draw. Text WIN to 80085 now!"]

new_text_vec = vectorizer.transform(new_text)

prediction = model.predict(new_text_vec)

if prediction[0] == 1:
    print("Spam XX")
else:
    print("Ham !!")

Spam XX


In [22]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [23]:
import sys
print(sys.executable)

/usr/bin/python3


In [24]:
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

In [25]:
nn_model = Sequential()

nn_model.add(Dense(128, activation='relu', input_shape=(X_train_dense.shape[1],)))
nn_model.add(Dense(64, activation='relu'))
nn_model.add(Dense(1, activation='sigmoid'))

nn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       993,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,001,601 (3.82 MB)

 Trainable params: 1,001,601 (3.82 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [27]:
nn_model.fit(
    X_train_dense, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.9050 - loss: 0.2716 - val_accuracy: 0.9821 - val_loss: 0.0650
Epoch 2/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.9903 - loss: 0.0338 - val_accuracy: 0.9888 - val_loss: 0.0441
Epoch 3/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9983 - loss: 0.0081 - val_accuracy: 0.9843 - val_loss: 0.0530
Epoch 4/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9998 - loss: 0.0023 - val_accuracy: 0.9843 - val_loss: 0.0646
Epoch 5/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 1.0000 - loss: 8.6625e-04 - val_accuracy: 0.9843 - val_loss: 0.0641


In [28]:
loss, acc = nn_model.evaluate(X_test_dense, y_test, verbose=0)
print("NN Accuracy of :", acc)

NN Accuracy of : 0.9793722033500671


In [29]:
new_msg_vec_dense = new_text_vec.toarray()

prediction = nn_model.predict(new_msg_vec_dense)

if prediction[0][0] > 0.5:
    print("Spam !!")
else:
    print("Ham !!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Spam !!


In [31]:
print(prediction)

[[0.9998614]]


In [33]:
nn_model.save("spam_classifier_nn.h5")

from tensorflow.keras.models import load_model
loaded_model = load_model("spam_classifier_nn.h5")

In [34]:
!pip install transformers

In [18]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="mrm8488/bert-tiny-finetuned-sms-spam-detection",
    return_all_scores=True
)

print(classifier(["Hey, are we still meeting tomorrow for coffee?"]))

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.9371647834777832}]


In [19]:
label_map = {
    "LABEL_0": "Ham",
    "LABEL_1": "Spam"
}

result = classifier(["Hey, are we still meeting tomorrow for coffee?"])

print(label_map[result[0]["label"]])

Ham


In [41]:
print(classifier.model.config.id2label)

{0: 'LABEL_0', 1: 'LABEL_1'}
